In [72]:
import  time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, ElementClickInterceptedException, StaleElementReferenceException


In [5]:
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
import re

In [90]:
s = Service('C:/Users/samir/Downloads/chromedriver-win64/chromedriver.exe')
driver = webdriver.Chrome(service=s)

driver.get('https://www.fotmob.com/players/737066/erling-haaland')
time.sleep(2)

pages = []   # NEW: store each page of matches
while True:
    # NEW: grab the current 10 matches before clicking Previous
    body = driver.find_element(By.CSS_SELECTOR, '[class*="PlayerMatchStatsTableBody"]')
    pages.append(body.get_attribute('outerHTML'))
    try:
        wait = WebDriverWait(driver, 2)
        button = wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="main-content"]/div[2]/div[1]/div[4]/div/div[2]/div[3]/div/button[1]')))
        button.click()
        time.sleep(1)
    except StaleElementReferenceException:
        continue
    except (TimeoutException, ElementClickInterceptedException):
        print("All items loaded!")
        break

html = "\n".join(pages)   # CHANGED: all pages instead of page_source
with open("haaland_stats.html", "w", encoding="utf-8") as file:
    file.write(html)
input("Press Enter in the terminal to close the browser...")
driver.quit()

All items loaded!


In [3]:
with open("haaland_stats.html", "r", encoding="utf-8") as file:
    html = file.read()

In [6]:
soup = BeautifulSoup(html, "html.parser")
soup.prettify()

container = soup.find_all('div', {'class': 'css-iho0mw-PlayerMatchStatsTableBody e2jthdr2'})

In [11]:
# variables to store the status
competitions = []
date= []
opponent_team = []
status = []
scoreline = []
own_team = []
minutes_played = []
position = []
goals = []
assists = []
yellow_cards = []
red_cards = []
rating = []


In [12]:
for items in container:
    each_match = items.find_all('a', {'class': re.compile('PlayerMatchStatsTableRow')})
    for detail in each_match:

        try:
            league_and_date = detail.find('div', {'class': re.compile('LeagueIconAndDate')})
        except:
            league_and_date = None

        try:
            competitions.append(league_and_date.find('span').text)
        except:
            competitions.append(np.nan)

        try:
            date.append(league_and_date.find('p').text)
        except:
            date.append(np.nan)

        try:
            opponent_team.append(
                detail.find('span', {'class': re.compile('TeamName')}).text
            )
        except:
            opponent_team.append(np.nan)

        try:
            result_div = detail.find('div', {'class': re.compile('Result')})
            spans = result_div.find_all('span', {'class': re.compile('grid-area')})
            status.append(
                [s.text for s in spans if 'invisible' not in ' '.join(s.get('class', []))][0]
            )
        except:
            status.append(np.nan)

        try:
            scoreline.append(
                detail.find('div', {'class': re.compile('Result')})
                      .find('span', {'class': re.compile('ScoreLine')}).text
            )
        except:
            scoreline.append(np.nan)

        #match status
        try:
            stats_wrapper = detail.find('div', {'class': re.compile('PlayerMatchStatsData')})
            match_stats = stats_wrapper.find_all('div', {'class': re.compile('DataCell')})
        except:
            match_stats = []

        try:
            own_team.append(
                match_stats[0].find('span').text
            )
        except:
            own_team.append(np.nan)

        try:
            minutes_played.append(
                match_stats[1].find('span').text
            )
        except:
            minutes_played.append(np.nan)

        try:
            position.append(
                match_stats[2].text
            )
        except:
            position.append(np.nan)

        try:
            goals.append(
                match_stats[3].text
            )
        except:
            goals.append(np.nan)

        try:
            assists.append(
                match_stats[4].text
            )
        except:
            assists.append(np.nan)

        try:
            yellow_cards.append(
                match_stats[5].text
            )
        except:
            yellow_cards.append(np.nan)

        try:
            red_cards.append(match_stats[6].text)
        except:
            red_cards.append(np.nan)

        try:
            rating.append(
               match_stats[7].text
            )
        except:
            rating.append(np.nan)

In [13]:
df = pd.DataFrame({
    'competition': competitions,
    'date': date,
    'opponent_team': opponent_team,
    'status': status,
    'scoreline': scoreline,
    'own_team': own_team,
    'minutes_played': minutes_played,
    'position': position,
    'goals': goals,
    'assists': assists,
    'yellow_cards': yellow_cards,
    'red_cards': red_cards,
    'rating': rating
})

In [14]:
    df.head(20)

,competition,date,opponent_team,status,scoreline,own_team,minutes_played,position,goals,assists,yellow_cards,red_cards,rating
0,Premier League,today,Fulham,D,1 - 1,Manchester City,90,ST,0,0,0,0,6.5
1,UEFA Nations League A,Nov 18,Denmark,W,2 - 3,Norway,90,ST,2,0,1,0,9.0
2,UEFA Nations League A,Nov 14,Wales,W,3 - 1,Norway,90,ST,1,1,0,0,8.5
3,Premier League,Nov 7,Nottm Forest,L,3 - 1,Manchester City,90,ST,0,0,0,0,6.7
4,Champions League,Nov 5,RB Leipzig,W,2 - 3,Manchester City,90,ST,1,0,0,0,8.0
5,Premier League,Oct 31,Brighton,L,0 - 1,Manchester City,90,ST,0,0,0,0,6.2
6,Premier League,Oct 24,Aston Villa,L,2 - 1,Manchester City,90,ST,1,0,0,0,7.5
7,Champions League,Oct 21,AEK Athens,W,5 - 1,Manchester City,0,-,0,0,0,0,-
8,Premier League,Oct 17,Ipswich,W,3 - 2,Manchester City,90,ST,2,0,0,0,9.0
9,Champions League,Oct 15,PSG,L,2 - 4,Manchester City,90,ST,2,0,0,0,9.1


In [15]:
df.to_csv("haaland_stats_uncleaned.csv")